# 13. PPU Detection - YoloV8N_PPU

| 항목 | 내용 |
|---|---|
| 대상 모델 | `YoloV8N_PPU.dxnn` |
| 카테고리 | **PPU Detection** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 32바이트 레코드의 자리별 dtype 을 밝히고 값을 직접 꺼내 해석할 수 있다.
2. 하드웨어가 후보를 먼저 걸러내는 구조의 이득을 바이트 수로 비교해 제시할 수 있다.
3. `parse_model` 이 보고하는 개수와 실제 받는 개수가 다른 이유를 설명할 수 있다.
4. 같은 장면을 12번과 비교해 두 구성의 차이를 결과와 함께 설명할 수 있다.

## 1. PPU Detection (하드웨어 후처리 검출)

**검출 후보를 걸러내는 계산까지 가속기 안에서 끝내고, 살아남은 것만 바이트 레코드로 받는 방식**

하는 일 자체는 객체 검출과 같다. 다른 것은 **어디까지가 장치의 일인가**다. 후보를 만들고 점수로 걸러내는
단계를 전용 회로(PPU)가 처리하므로, 프로그램은 이미 줄어든 목록을 받아 겹침만 정리한다.

**출력의 성격**

숫자 배열이 아니라 **정해진 자리에 여러 종류의 값이 섞여 있는 바이트 묶음**이다. 자리마다 형식이 달라 순서를
모르면 읽을 수 없고, 개수도 실행할 때마다 달라진다.

**활용 사례**

- 전송량과 Host 계산을 줄여야 하는 임베디드 검출
- 여러 대의 카메라를 한 보드에서 처리
- 후처리 코드를 최소화한 제품 구성

**포함되지 않는 작업**

- 부류 점수 전체를 살펴보는 일 (걸러진 뒤라 남아 있지 않음)
- 픽셀 단위 영역 구분 (Instance Segmentation)
- 대상 추적 (별도 알고리즘의 일)

## 2. 전체 파이프라인

```text
sample_street.jpg  768×576 BGR
  → 전처리         BGR→RGB · 비율 유지 축소 640×480 · 위아래 여백 80 → 640×640
  → run()          입력 Tensor 하나를 NPU 에 전달
  → npu_0 (PPU)    BBOX  dtype BBOX · 레코드당 32바이트  (Public Output)
  → 후처리         자리별 dtype 으로 값 꺼내기 → 점수 임계값 → 중심좌표를 모서리로 → NMS
  → Result         DetectionResult (box · confidence · class_id)
  → 시각화         사각형과 `이름: 점수` 라벨을 원본 위에 그림
```

![13. PPU Detection 전체 파이프라인](resources/13_PPU_Detection_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `images` |
| Shape / Dtype | `[1, 640, 640, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **Letterbox — 비율 유지 후 여백 채우기** |
| Normalization | 없음 |

입력은 12번과 같다. 좌표를 답하므로 비율을 유지하고, `gain`·`pad_x`·`pad_y` 를 Context 에 담아 둔다. 이번 실행의
`gain` 은 `0.833333`, 위아래 여백은 80 씩이다.

### 3.2 출력

| Tensor | Shape / dtype | 의미 | Public |
|---|---|---|---|
| `BBOX` | `[8400]` / `BBOX` | 살아남은 검출 레코드 목록 | O |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. **Task 는 `npu_0` 하나뿐이며 CPU Task
가 없다** — 걸러내기까지 장치 안에서 끝나므로 CPU 로 넘겨 다시 계산할 것이 없다.

**dtype 이 숫자 형식이 아니다.** `float32` 나 `uint8` 이 아니라 `BBOX` 라고 적혀 있고, 요소 하나가 32바이트다.
이는 한 검출의 정보를 정해진 자리에 나란히 담은 **구조체**이며, 자리마다 형식이 다르다.

| 바이트 | 개수 | 형식 | 내용 |
|---|---:|---|---|
| 0~15 | 4 | `float32` | 중심 좌표와 크기 `x, y, w, h` |
| 16~19 | 4 | `uint8` | `grid_y`, `grid_x`, `box_idx`, `layer_idx` |
| 20~23 | 1 | `float32` | 점수 |
| 24~27 | 1 | `uint32` | 부류 번호 |
| 28~31 | 4 | — | 채움(padding) |

**바이트를 임의로 나누면 값이 망가진다.** 32바이트를 `float32` 8개로 한꺼번에 읽으면 점수 자리까지 실수로
해석되고, 부류 번호는 아주 큰 실수로 보인다. 자리마다 **그 자리의 형식으로 다시 보는** 방법을 써야 한다.

**개수가 정해져 있지 않다.** `parse_model` 은 `[8400]` 을 보고하는데 이는 **담을 수 있는 최대 개수**이고, 실제로
받는 것은 장치가 걸러 낸 뒤 남은 개수다. 이번 실행에서는 수백 개가 왔으며 정확한 개수는 아래 실행 셀 출력에서
확인한다. **같은 사진을 다시 넣어도 개수가 조금씩 달라진다** — 이 보드에서 284개와 289개를 관찰했다. 그래서 코드는
정해진 개수를 가정하지 않고 받은 개수를 그대로 세어 쓴다.

**12번과 견주면 이 구조의 이득이 드러난다.**

| | 12번 (CPU Task 가 정리) | 13번 (PPU 가 정리) |
|---|---:|---:|
| 받는 형태 | `[1, 25200, 85]` `float32` | 32바이트 레코드 수백 개 |
| 바이트 | 8,568,000 | 약 9,000 |
| Host 가 할 일 | obj 걸러내기 · 부류 최고점 · NMS | NMS |

**900배 넘게 작다.** 후보를 만들어 놓고 밖에서 버리는 대신, 버릴 것을 장치 안에서 버리고 남은 것만 내보내기
때문이다.

**YOLOv8 은 앵커를 쓰지 않는다.** 그래서 레코드의 `x, y, w, h` 가 격자·앵커로 되돌릴 값이 아니라 **입력 640
기준의 중심 좌표와 크기 그대로**다. 이번 실행에서 값의 범위는 `7.38`~`619.44` 로 640 안에 들어온다.
레코드의 `grid` 네 값은 앵커를 쓰는 다른 PPU 모델을 위한 자리이며 이 모델의 후처리는 읽지 않는다.

**80개 부류의 번호는 12번과 같은 COCO 목록**을 쓴다. 번호의 정본은 실행 코드가 들고 있는 이름 배열이며,
공식의 띄엄띄엄한 1~90 을 0~79 로 눌러 담은 관행이다.

| index | 이름 | 고른 이유 |
|---:|---|---|
| 0 | person | 목록의 첫 항목 |
| 1 | bicycle | 이번 실행 검출 |
| 2 | car | 이번 실행 검출 |
| 3 | motorcycle | 1·2 와 이어지는 교통 부류 |
| 9 | traffic light | 레코드 예시의 부류 번호 |
| 16 | dog | 이번 실행 1위 |
| 39 | bottle | 중간 자리 확인용 |
| 56 | chair | 중간 자리 확인용 |
| 72 | refrigerator | 이번 실행 레코드의 최대 번호 |
| 79 | toothbrush | 목록의 마지막 항목 |

전체 목록: [ultralytics coco.yaml](https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco.yaml)

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 출력 dtype 이 `BBOX`,
요소 크기가 32바이트이고 CPU Task 가 없는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "YoloV8N_PPU.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `score_threshold` | 0.4 | 0.4 | 이 점수에 못 미치는 레코드를 버림 |
| `nms_threshold` | 0.5 | 0.5 | 겹침이 이 비율을 넘으면 낮은 점수를 버림 |

**이 후처리는 `conf_threshold` 를 먼저 찾고 없으면 `score_threshold` 를 본다.** 그래서 파일의 `0.4` 가 그대로
적용된다. 12번은 `conf_threshold` 만 찾아 파일의 `score_threshold` 가 무시됐는데, **같은 계열이라도 후처리마다
읽는 key 가 다르다**는 점을 함께 보아야 한다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 설정 파일과 COCO 80개 이름을 읽는다. 이후 셀은 여기서 만든
이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_street.jpg"
CONFIG_PATH = DX_APP / "src" / "python_example" / "ppu" / "yolov8n_ppu" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.utility import get_labels  # noqa: E402

COCO_LABELS = get_labels("coco80")
RECORD_BYTES = 32

print("config", CONFIG)
print("classes", len(COCO_LABELS), "/ index 16 =", COCO_LABELS[16])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. `scale`(gain)·`pad_x`·`pad_y` 가
  좌표 복원의 근거다
- `DetectionResult` — 후처리가 만들어 시각화로 넘기는 결과. 사각형·점수·부류 번호를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale: float = 1.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class DetectionResult:
    """대상 하나의 사각형·점수·부류. 원본 대응: common/base/i_processor.py"""

    box: list = field(default_factory=list)
    confidence: float = 0.0
    class_id: int = 0
    class_name: str = ""


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력 dtype 이 `BBOX` 이고 요소 크기가 32인 것을
확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")
print(f"레코드 하나 = {output_info[0]['elem_size']} 바이트 / 최대 {output_info[0]['shape'][0]}개")

### 4.2 전처리

12번과 같은 letterbox 를 쓴다. 비율을 유지해 640×480 으로 줄이고 위아래에 80씩 회색 여백을 넣는다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 640×640, 그중 실제 사진은 640×480
- Context — `scale`(gain) 과 `pad_x`·`pad_y`

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 12번과 같은 사진이므로 결과를 나란히 비교할 수 있다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(10, 7))

#### 전처리 함수 정의

- `ppu_preprocessor_create` — 모델 입력 크기와 여백 색을 state 에 담음
- `ppu_letterbox` — 비율 유지 축소와 여백 채우기
- `ppu_preprocessor_process` — letterbox 를 적용하고 복원용 값을 Context 에 담음

In [ ]:
def ppu_preprocessor_create(input_width: int, input_height: int,
                            pad_color: tuple = (114, 114, 114)) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height, "pad_color": pad_color}


def ppu_letterbox(state: dict, image: np.ndarray) -> tuple:
    """비율을 유지해 줄이고 남는 자리를 채운다. 원본 대응: LetterboxPreprocessor._letterbox()"""
    height, width = image.shape[:2]
    gain = min(state["input_height"] / height, state["input_width"] / width)
    new_size = int(round(width * gain)), int(round(height * gain))

    delta_w = (state["input_width"] - new_size[0]) / 2
    delta_h = (state["input_height"] - new_size[1]) / 2
    if (width, height) != new_size:
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(delta_h - 0.1)), int(round(delta_h + 0.1))
    left, right = int(round(delta_w - 0.1)), int(round(delta_w + 0.1))
    padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT,
                                value=state["pad_color"])
    return padded, gain, (top, left)


def ppu_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    padded, gain, pad = ppu_letterbox(state, rgb)
    ctx.scale = gain
    ctx.pad_y, ctx.pad_x = pad
    return padded, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다.

In [ ]:
preprocessor_state = ppu_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = ppu_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"gain={ctx.scale:.6f} pad_x={ctx.pad_x} pad_y={ctx.pad_y}")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이 포함돼
있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"{array.nbytes:,} bytes")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- dtype — 바이트(`uint8`) 배열로 들어온다
- 레코드 개수 — `parse_model` 이 보고한 최대 개수보다 훨씬 적다
- 한 레코드의 32바이트 — 자리별 형식으로 읽어야 뜻이 생긴다
- 잘못 읽으면 어떻게 되는지 — 32바이트를 전부 `float32` 로 보면 점수·부류가 망가진다

In [ ]:
raw = np.asarray(outputs[0])
records = raw[0]

print(f"raw           shape={raw.shape} dtype={raw.dtype} {raw.nbytes:,} bytes")
print(f"레코드         {records.shape[0]}개 × {records.shape[1]}바이트 "
      f"(최대 {output_info[0]['shape'][0]}개 중)")
print(f"\n첫 레코드 32바이트:\n  {records[0].tolist()}")

wrong = records[0].view(np.float32)
print(f"\n전부 float32 로 잘못 읽으면: {np.array2string(wrong, precision=3)}")
print("  → 앞 넷은 좌표로 맞지만 점수·부류 자리가 뜻 없는 값이 된다")

### 4.4 후처리

바이트 레코드를 자리별 형식으로 읽어 값을 꺼내고, 점수로 걸러 내고, 중심 좌표를 모서리로 바꾸어 겹침을 정리한 뒤
좌표를 원본으로 되돌린다.

**자리별로 형식을 바꿔 보는 것이 이 후처리의 핵심이다.** `view()` 는 같은 바이트를 다른 형식으로 다시 보는
것이며 값을 바꾸지 않는다. 0~15바이트는 `float32` 넷, 20~23바이트는 `float32` 하나, 24~27바이트는 `uint32`
하나로 본다. **자리를 하나만 밀려도 전혀 다른 값이 나온다.**

**앵커·격자 복원 단계가 없다.** YOLOv8 은 앵커를 쓰지 않으므로 레코드의 `x, y, w, h` 가 이미 입력 좌표계의
중심과 크기다. 12번이 `(sigmoid(t) × 2 - 0.5 + grid) × stride` 를 계산해야 했던 자리가 이 모델에는 없다.

**점수 걸러내기가 두 번 나타난다.** 먼저 배열 연산으로 임계값을 넘는 레코드만 남기고, 그다음 겹침 정리 함수에도
같은 임계값을 넘긴다. 앞의 것이 계산량을 줄이고 뒤의 것이 최종 판정을 한다.

#### 후처리 함수 정의

- `ppu_postprocessor_create` — 두 임계값을 state 에 담음
- `ppu_parse_records` — 32바이트 레코드에서 좌표·점수·부류를 자리별 형식으로 꺼냄
- `ppu_postprocessor_process` — 걸러내기·NMS·좌표 복원을 거쳐 Result 생성

In [ ]:
def ppu_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()

    conf_threshold 를 먼저 찾고 없으면 score_threshold 를 본다. 이 모델은 파일의 값이 그대로 쓰인다.
    """
    return {
        "score_threshold": config.get("conf_threshold", config.get("score_threshold", 0.4)),
        "nms_threshold": config.get("nms_threshold", 0.5),
    }


def ppu_parse_records(records: np.ndarray) -> tuple:
    """32바이트 레코드를 자리별 형식으로 읽어 좌표·점수·부류를 돌려준다.

    원본 대응: YOLOv8PPUPostprocessor.process() 의 필드 파싱
    """
    boxes = records[:, :16].view(np.float32).reshape(-1, 4)   # x, y, w, h
    scores = records[:, 20:24].view(np.float32).flatten()
    labels = records[:, 24:28].view(np.uint32).flatten()
    return boxes, scores, labels


def ppu_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    records = np.asarray(outputs[0])[0]
    if records.ndim != 2 or records.shape[1] != RECORD_BYTES or records.shape[0] == 0:
        return []

    boxes, scores, labels = ppu_parse_records(records)

    kept = scores >= state["score_threshold"]
    if not np.any(kept):
        return []
    boxes, scores, labels = boxes[kept], scores[kept], labels[kept]

    # 앵커가 없으므로 중심 좌표와 크기를 그대로 모서리로 바꾼다.
    corners = np.column_stack([
        boxes[:, 0] - boxes[:, 2] * 0.5, boxes[:, 1] - boxes[:, 3] * 0.5,
        boxes[:, 0] + boxes[:, 2] * 0.5, boxes[:, 1] + boxes[:, 3] * 0.5,
    ])
    boxes_xywh = np.column_stack([corners[:, 0], corners[:, 1],
                                  corners[:, 2] - corners[:, 0], corners[:, 3] - corners[:, 1]])
    indices = cv2.dnn.NMSBoxes(boxes_xywh.tolist(), scores.tolist(),
                               state["score_threshold"], state["nms_threshold"])
    if len(indices) == 0:
        return []

    gain = max(ctx.scale, 1e-6)
    results = []
    for index in np.array(indices).reshape(-1):
        box = corners[index]
        results.append(DetectionResult(
            box=[float(np.clip((box[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
                 float(np.clip((box[2] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[3] - ctx.pad_y) / gain, 0, ctx.original_height - 1))],
            confidence=float(scores[index]),
            class_id=int(labels[index]),
        ))
    return results

#### 후처리 실행

확인할 것:

- 자리별로 읽은 값 — 좌표는 640 안, 점수는 0~1, 부류는 0~79
- 단계별 개수 — 받은 레코드 → 점수 통과 → 최종
- 각 결과의 부류·점수·사각형 — 원본 크기 안에 들어오는지

In [ ]:
postprocessor_state = ppu_postprocessor_create(CONFIG)
boxes, scores, labels = ppu_parse_records(records)

print(f"좌표 x,y,w,h   {boxes.min():8.2f} ~ {boxes.max():8.2f} (입력 {INPUT_WIDTH} 안)")
print(f"점수           {scores.min():8.6f} ~ {scores.max():8.6f}")
print(f"부류 번호       {labels.min()} ~ {labels.max()}")
print(f"\n첫 레코드 해석  x,y,w,h={np.array2string(boxes[0], precision=2)} "
      f"score={scores[0]:.6f} label={labels[0]} ({COCO_LABELS[labels[0]]})")

results = ppu_postprocessor_process(postprocessor_state, outputs, ctx)
score_pass = int((scores >= postprocessor_state["score_threshold"]).sum())
print(f"\n받은 레코드 {records.shape[0]}개 → 점수 {postprocessor_state['score_threshold']} 통과 "
      f"{score_pass}개 → NMS {postprocessor_state['nms_threshold']} 통과 {len(results)}개")

for order, det in enumerate(sorted(results, key=lambda r: -r.confidence), start=1):
    x1, y1, x2, y2 = det.box
    print(f"  #{order} {COCO_LABELS[det.class_id]:12s} score={det.confidence:.4f} "
          f"box=({x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f})")

### 4.5 시각화

#### Visualizer 코드

12번과 같은 시각화를 쓴다. 결과 객체만 받아 사각형과 `이름: 점수` 라벨을 원본 위에 그리며, 부류별 색은 난수
씨앗을 고정해 실행마다 같게 만든다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글 자모가 없어
한글을 넣으면 글자가 깨진다.

In [ ]:
def ppu_visualizer_create(labels: list, seed: int = 0) -> dict:
    """부류별 색을 만들어 state 에 담는다. 원본 대응: Factory.create_visualizer()"""
    generator = np.random.default_rng(seed)
    return {"labels": labels, "palette": generator.uniform(0, 255, size=(len(labels), 3))}


def ppu_visualizer_visualize(state: dict, image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()

    for det in results:
        x1, y1, x2, y2 = [int(value) for value in det.box]
        color = state["palette"][det.class_id % len(state["palette"])]
        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

        name = (state["labels"][det.class_id] if det.class_id < len(state["labels"])
                else f"class_{det.class_id}")
        label = f"{name}: {det.confidence:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        label_y = y1 - 10 if y1 - 10 > label_h else y1 + 10
        cv2.rectangle(output, (x1, label_y - label_h), (x1 + label_w, label_y + label_h),
                      color, cv2.FILLED)
        cv2.putText(output, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1,
                    cv2.LINE_AA)

    return output

#### 결과 시각화

검출 결과를 inline 으로 표시한다.

In [ ]:
visualizer_state = ppu_visualizer_create(COCO_LABELS)
output_image = ppu_visualizer_visualize(visualizer_state, image, results)
notebook_show_bgr(output_image, f"{len(results)} objects detected (PPU)", figsize=(11, 8))

#### 받은 레코드 전부를 점수 순으로 보기

장치가 이미 걸러 낸 뒤이지만 아직 점수가 낮은 레코드도 섞여 있다. 점수 분포를 구간별로 세어 보면 임계값 `0.4`
가 어디를 자르는지, 그리고 남은 레코드 대부분이 낮은 점수라는 사실이 드러난다.

In [ ]:
edges = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
print(f"받은 레코드 {records.shape[0]}개의 점수 분포")
for low, high in zip(edges[:-1], edges[1:]):
    count = int(((scores >= low) & (scores < high)).sum())
    bar = "#" * min(40, count)
    print(f"  {low:.1f} ~ {high:.1f}  {count:4d}  {bar}")
print(f"임계값 {postprocessor_state['score_threshold']} 이상 = {score_pass}개, "
      f"그중 겹침 정리 뒤 {len(results)}개")